# 02 — Ingestion

**Tasks:** M1-T05 (pilot extraction). The scaled extraction (M1-T06) will be added here.

**Purpose:** a working, cached, provenance-tracked composite download; the real processing-unit (PU) cost of composites; and three data checks: cloud-mask holes over bright surfaces, a plausible reflectance range, and harmonisation across the 2022-01-25 processing-baseline change.

**Inputs:** `config/project.yaml` (`pilot` section), `config/aois.geojson`, `reports/results/m1_aoi_availability.csv`, `.env` (credentials).

**Outputs:** composites in `data/raw/<aoi_id>/` and statistics in `data/raw/stats/` (local, with provenance sidecars); `reports/results/m1_pilot_cost.csv`, `m1_pilot_qa.csv`, `m1_harmonisation.csv`; figures `reports/figures/m1_pilot_composite.png`, `m1_harmonisation.png`; a row in `data/api_usage_log.csv`.

**Run time:** under a minute; the first run makes a few API requests (plan printed below), and later runs use the cache and make none.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import logging
import platform
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import pandas as pd
import rasterio
import sentinelhub

import desertification
from desertification.aoi import season_window, sub_window
from desertification.config import get_path, load_config, load_env
from desertification.feasibility import composite_pu, grid_shape, season_fraction
from desertification.features import ndvi
from desertification.ingest import download_composite, fetch_reflectance_stats
from desertification.qa import (
    harmonisation_ok, parse_stats_response, reflectance_summary, valid_count_summary,
    year_over_year_step,
)
from desertification.sh_client import missing_credentials
from desertification.usage import (
    append_usage, check_cap, estimate_process_pu, latest_balance, make_entry, read_usage_log,
)
from desertification.viz import plot_pilot_panels, plot_reflectance_series

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
cfg = load_config()
SEED = cfg["project"]["seed"]
rng = np.random.default_rng(SEED)
pilot = cfg["pilot"]
RESULTS, FIGURES = get_path(cfg, "results"), get_path(cfg, "figures")
print(f"Python {platform.python_version()} | desertification {desertification.__version__} | "
      f"sentinelhub {sentinelhub.__version__} | rasterio {rasterio.__version__}")

Python 3.12.3 | desertification 0.1.0 | sentinelhub 3.12.1 | rasterio 1.5.2


## Pilot extraction (M1-T05)

**Design.** One AOI (`pilot.aoi_id`: the Aral Sea, the brightest surface and the one with the most acquisitions, so an upper bound on cost and a hard test for the cloud mask) and one season (the dry season of `pilot.year`). The request covers a centred 5 km × 5 km window rather than the full 20 km square: PU cost scales linearly with area (CDSE rules), and a full-size composite of the original design would exceed the whole setup-and-pilot budget.

Two designs, both 16-bit integer output (reflectance × 10 000; Decision log #45):

- **b10**: all 10 bands in `sentinel2.bands` + SCL (matches the current §7.2 schema);
- **c6**: only the six bands the nine indices need (B02 B03 B04 B08 B11 B12) + SCL.

Each composite is the per-pixel median of every cloud-free acquisition in the season (SCL mask from config), plus a count of valid acquisitions. A small Statistical API time series over the stable erg checks harmonisation.

**Quota.** The plan below prints the number of requests and the projected PU before anything is sent. Set `ALLOW_API_CALLS = True` only after it has been approved.

In [2]:
ALLOW_API_CALLS = False

aois = {f["properties"]["aoi_id"]: f["properties"] for f in json.loads(get_path(cfg, "aois").read_text())["features"]}
aoi = aois[pilot["aoi_id"]]
window = sub_window(aoi["bbox_utm"], pilot["window_m"], cfg["aoi_selection"]["snap_m"])
season_months = aoi[f"{pilot['season']}_season_months"]
start, end = season_window(season_months, pilot["year"])
width, height = grid_shape(window, pilot["resolution_m"])

avail = pd.read_csv(RESULTS / "m1_aoi_availability.csv")
n_dates = int(avail.query("aoi_id == @pilot['aoi_id'] and year == @pilot['year']")["n_dates"].item())
samples = n_dates * season_fraction(season_months)

plan = [{"request": f"composite {name}", "bands": len(b), "projected_pu":
         round(composite_pu(width, height, len(b) + 1, samples, float32=False), 1)}
        for name, b in pilot["designs"].items()]
h = pilot["harmonisation"]
erg = aois[h["aoi_id"]]
h_window = sub_window(erg["bbox_utm"], h["window_m"], cfg["aoi_selection"]["snap_m"])
hw, hh = grid_shape(h_window, h["resolution_m"])
n_intervals = 24
plan.append({"request": "statistics (harmonisation)", "bands": len(h["bands"]),
             "projected_pu": round(n_intervals * estimate_process_pu(hw, hh, len(h["bands"]) + 1, statistical=True), 2)})
plan = pd.DataFrame(plan)
total_pu = plan["projected_pu"].sum()
fits, left = check_cap(read_usage_log(get_path(cfg, "api_usage_log")), "setup_pilot", total_pu, cfg["quota"]["pu_caps"])
print(f"AOI {pilot['aoi_id']}, {pilot['season']} season {start}..{end}, window {width} x {height} px at {pilot['resolution_m']} m")
print(f"Acquisition dates in {pilot['year']}: {n_dates} -> about {samples:.0f} samples in the season")
print(f"Plan: {len(plan)} requests (+1 OAuth token), projected ~{total_pu:.0f} PU "
      f"(fits setup_pilot cap: {fits}; PU left in it afterwards: {left:.0f})")
plan

AOI aral, dry season 2023-06-01..2023-09-30, window 250 x 250 px at 20 m
Acquisition dates in 2023: 145 -> about 48 samples in the season
Plan: 3 requests (+1 OAuth token), projected ~69 PU (fits setup_pilot cap: True; PU left in it afterwards: 907)


,request,bands,projected_pu
0,composite b10,10,42.0
1,composite c6,6,26.7
2,statistics (harmonisation),4,0.4


In [3]:
load_env(cfg)
missing = missing_credentials()
go = ALLOW_API_CALLS and not missing and fits
print("Credentials present:", not missing, "| API calls allowed:", go)

composites, measured = {}, []
for name, bands in pilot["designs"].items():
    path, called, reported = download_composite(
        cfg, pilot["aoi_id"], window, aoi["utm_epsg"], start, end, pilot["resolution_m"],
        bands=bands, label=f"pilot_{name}_w{pilot['window_m'] // 1000}km", allow_api=go,
    )
    composites[name] = path
    measured.append({"request": f"composite {name}", "called_api": called, "reported_pu": reported,
                     "file": None if path is None else path.relative_to(cfg["_root"]).as_posix()})

stats_data, _, stats_called = fetch_reflectance_stats(
    cfg, f"{h['aoi_id']}_harmonisation", h_window, erg["utm_epsg"], h["start"], h["end"],
    h["interval"], h["resolution_m"], h["bands"], allow_api=go,
)
measured.append({"request": "statistics (harmonisation)", "called_api": stats_called, "reported_pu": None,
                 "file": None})
n_calls = sum(m["called_api"] for m in measured)
print(f"API requests made in this run: {n_calls}")
pd.DataFrame(measured)

Credentials present: False | API calls allowed: False
API requests made in this run: 0


,request,called_api,reported_pu,file
0,composite b10,False,None,None
1,composite c6,False,None,None
2,statistics (harmonisation),False,None,None


### Quota ledger and measured cost

Copy the dashboard balances from before and after the run. If the service reported PU per request in a response header, it is shown above (`reported_pu`) and stored in each file's sidecar; otherwise the dashboard difference is the measured total.

In [4]:
PU_BEFORE, PU_AFTER = None, None
REQUESTS_BEFORE, REQUESTS_AFTER = None, None

log_path = get_path(cfg, "api_usage_log")
if n_calls:
    append_usage(log_path, make_entry(
        task_id="M1-T05", api="process+statistical", budget_category="setup_pilot",
        request_count=n_calls + 1,  # + OAuth token
        purpose=f"Pilot: {n_calls} request(s) ({', '.join(m['request'] for m in measured if m['called_api'])})",
        pu_before=PU_BEFORE, pu_after=PU_AFTER, requests_before=REQUESTS_BEFORE, requests_after=REQUESTS_AFTER,
    ))
    print("Logged in", log_path.relative_to(cfg["_root"]))
print("Latest recorded balance:", latest_balance(read_usage_log(log_path)))

cost = plan.merge(pd.DataFrame(measured)[["request", "reported_pu"]], on="request")
if PU_BEFORE is not None and PU_AFTER is not None and n_calls:
    cost["dashboard_total_pu"] = PU_BEFORE - PU_AFTER
if any(p is not None for p in composites.values()):
    cost.to_csv(RESULTS / "m1_pilot_cost.csv", index=False)
cost

Latest recorded balance: {'pu': 29976.0, 'requests': 29849.0}


,request,bands,projected_pu,reported_pu
0,composite b10,10,42.0,None
1,composite c6,6,26.7,None
2,statistics (harmonisation),4,0.4,None


## Data checks

**Cloud-mask holes (gotcha #2).** Bright surfaces such as salt crust can be misclassified as cloud by the SCL. The valid-count band shows how many acquisitions survived the mask at each pixel; large areas with 0 or very few would mean holes.

**Reflectance range.** Values are DN / 10 000; valid pixels should lie in 0–1 (bright salt can approach 1).

In [5]:
qa_rows = []
for name, path in composites.items():
    if path is None:
        continue
    with rasterio.open(path) as src:
        arr = src.read()
        names = list(src.descriptions)
    count = arr[names.index("valid_count")]
    vc = valid_count_summary(count)
    qa_rows.append({"design": name, "band": "valid_count", **vc})
    for b in [n for n in names if n != "valid_count"]:
        qa_rows.append({"design": name, "band": b, **reflectance_summary(arr[names.index(b)], count > 0)})
qa = pd.DataFrame(qa_rows)
if not qa.empty:
    qa.to_csv(RESULTS / "m1_pilot_qa.csv", index=False)
qa

""


In [6]:
path = composites.get("b10") or composites.get("c6")
if path is not None:
    with rasterio.open(path) as src:
        arr = src.read().astype(float)
        names = list(src.descriptions)
    band = lambda b: arr[names.index(b)] / 10_000
    count = arr[names.index("valid_count")]
    rgb = np.dstack([band("B04"), band("B03"), band("B02")]) * 2.5
    nd = np.where(count > 0, ndvi(band("B08"), band("B04")), np.nan)
    fig = plot_pilot_panels(
        rgb, nd, count,
        f"Pilot composite: {pilot['aoi_id']}, {pilot['season']} season {pilot['year']} (median of cloud-free acquisitions)",
        f"Contains modified Copernicus Sentinel data {pilot['year']}, processed by Sentinel Hub. "
        f"{pilot['window_m'] // 1000} km x {pilot['window_m'] // 1000} km window, {pilot['resolution_m']} m pixels.",
        out_path=FIGURES / "m1_pilot_composite.png",
    )

**Harmonisation (gotcha #1).** Sentinel Hub harmonises the January 2022 processing-baseline change by default. Over the stable erg, each month from February 2022 onward is compared with the same month a year earlier; an unharmonised offset would show as a drop of about 0.1 reflectance in every band, harmonised data as no systematic step.

In [7]:
if stats_data is not None:
    series = parse_stats_response(stats_data)
    steps = year_over_year_step(series, h["cutoff"])
    steps["ok"] = steps["median_step"].abs() <= h["tolerance"]
    steps.to_csv(RESULTS / "m1_harmonisation.csv", index=False)
    print("Harmonisation check passed:", harmonisation_ok(steps, h["tolerance"]))
    fig = plot_reflectance_series(
        series, h["cutoff"], f"Monthly reflectance over the stable erg ({h['window_m'] // 1000} km window)",
        f"Contains modified Copernicus Sentinel data {h['start'][:4]}-{h['end'][:4]}, processed by Sentinel Hub. "
        "Least-cloudy mosaic per month; cloud-masked with SCL.",
        out_path=FIGURES / "m1_harmonisation.png",
    )
    display(steps)
else:
    print("No statistics yet (API calls off or not cached).")

No statistics yet (API calls off or not cached).


### What to check

- **Cache:** re-run the notebook; `API requests made in this run` must be 0.
- **Holes:** `share_zero` of `valid_count` near 0, with no large blank patches in the count panel.
- **Range:** reflectance `p99` ≤ 1 and `share_above_1` near 0 for every band.
- **Harmonisation:** `|median_step|` ≤ 0.02 in every band.
- **Cost:** compare measured PU (header or dashboard) with `projected_pu`; the ratio calibrates the projections in `m1_pu_projection.csv`.